# Ch.16 — TensorBoard

> **Running theme:** The Ch.5 neural network converged — but what happened inside? TensorBoard instruments the training loop and makes the internals visible: loss curves, weight histograms, gradient distributions, and embedding projections. This chapter adds TensorBoard to the California Housing model and reads every panel.

## Core Idea

TensorBoard reads event files written by TF/PyTorch and renders:

| Panel | Logging trigger | Primary diagnostic |
|---|---|---|
| Scalars | `update_freq='epoch'` | Overfitting gap, convergence rate |
| Histograms | `histogram_freq=1` | Vanishing/exploding gradients |
| Projector | manual `np.savetxt` | Are learned embeddings meaningful? |
| Graphs | `write_graph=True` | Architecture verification |

```
tensorboard --logdir logs/
```
Then open `http://localhost:6006`

## Running Example

Dataset: **California Housing** (sklearn).  
Model: 3-layer dense network (same as Ch.5).  
Goal: instrument training with TensorBoard and read each diagnostic panel.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import datetime
import tensorflow as tf
from tensorflow import keras
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print(f"TensorFlow version: {tf.__version__}")

# ── Data ──────────────────────────────────────────────────────────────────────
housing    = fetch_california_housing()
X, y_raw   = housing.data, housing.target
y          = y_raw.reshape(-1, 1)

scaler = StandardScaler()
X_sc   = scaler.fit_transform(X)

X_tr, X_te, y_tr, y_te = train_test_split(X_sc, y,   test_size=0.2, random_state=42)
X_tr, X_va, y_tr, y_va = train_test_split(X_tr, y_tr, test_size=0.2, random_state=42)

print(f"Train: {X_tr.shape[0]:,}  Val: {X_va.shape[0]:,}  Test: {X_te.shape[0]:,}")

## Build the Model (Ch.5 Architecture)

In [ ]:
def build_model(learning_rate=1e-3, use_batchnorm=False):
    layers = [
        keras.layers.Input(shape=(X_tr.shape[1],)),
        keras.layers.Dense(64, activation='relu', name='hidden_1',
                           kernel_initializer='he_normal'),
    ]
    if use_batchnorm:
        layers.append(keras.layers.BatchNormalization())
    layers += [
        keras.layers.Dense(32, activation='relu', name='hidden_2',
                           kernel_initializer='he_normal'),
        keras.layers.Dense(16, activation='relu', name='hidden_3',
                           kernel_initializer='he_normal'),
        keras.layers.Dense(1,                     name='output'),
    ]
    model = keras.Sequential(layers)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='mse',
        metrics=['mae']
    )
    return model

model = build_model()
model.summary()

## TensorBoard Callback: Scalars + Histograms

The `TensorBoard` callback writes event files to `log_dir`.  
After training, open a terminal and run:  
```
tensorboard --logdir logs/
```
then navigate to `http://localhost:6006`.

In [ ]:
run_id  = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
log_dir = os.path.join('logs', f'ch16_{run_id}')

tb_callback = keras.callbacks.TensorBoard(
    log_dir        = log_dir,
    histogram_freq = 1,          # weight + bias histograms per epoch
    write_graph    = True,       # log computational graph (once)
    update_freq    = 'epoch',    # scalars per epoch — NOT per batch
    profile_batch  = 0,          # disable profiling (set to 2 to profile batch 2)
)

history = model.fit(
    X_tr, y_tr,
    validation_data = (X_va, y_va),
    epochs          = 60,
    batch_size      = 256,
    callbacks       = [tb_callback],
    verbose         = 0
)

print(f"Training complete. Logs: {log_dir}")
print(f"Run:  tensorboard --logdir logs/")

## Reading the Scalars Panel (in Notebook)

We can also plot the logged metrics directly from the `history` object.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

epochs = range(1, len(history.history['loss']) + 1)

ax1.plot(epochs, history.history['loss'],     label='train MSE', lw=1.5)
ax1.plot(epochs, history.history['val_loss'], label='val MSE',   lw=1.5, linestyle='--')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('MSE')
ax1.set_title('Loss (MSE) curves')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(epochs, history.history['mae'],     label='train MAE', lw=1.5)
ax2.plot(epochs, history.history['val_mae'], label='val MAE',   lw=1.5, linestyle='--')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('MAE')
ax2.set_title('MAE curves')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.suptitle('Scalars panel — same curves visible in TensorBoard', y=1.02)
plt.tight_layout(); plt.show()

final_val_rmse = history.history['val_loss'][-1] ** 0.5
print(f"Final validation RMSE: {final_val_rmse:.4f}")

## Custom Scalars: Learning Rate Logger

The TensorBoard callback doesn't log LR automatically. Write a custom callback using `tf.summary`.

In [ ]:
class LRSchedulerLogger(keras.callbacks.Callback):
    """Logs current learning rate to TensorBoard Scalars."""
    def __init__(self, log_dir):
        super().__init__()
        self.writer = tf.summary.create_file_writer(os.path.join(log_dir, 'custom'))

    def on_epoch_end(self, epoch, logs=None):
        lr = float(self.model.optimizer.learning_rate)
        with self.writer.as_default():
            tf.summary.scalar('learning_rate', lr, step=epoch)
        self.writer.flush()


# ── Model with cosine-decay LR schedule ──────────────────────────────────────
run_id2  = datetime.datetime.now().strftime('%Y%m%d_%H%M%S') + '_cosine'
log_dir2 = os.path.join('logs', f'ch16_{run_id2}')

cosine_lr  = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1e-3, decay_steps=60 * (len(X_tr) // 256)
)
model2 = build_model()
model2.compile(optimizer=keras.optimizers.Adam(cosine_lr), loss='mse', metrics=['mae'])

hist2 = model2.fit(
    X_tr, y_tr,
    validation_data = (X_va, y_va),
    epochs          = 60,
    batch_size      = 256,
    callbacks       = [
        keras.callbacks.TensorBoard(log_dir=log_dir2, histogram_freq=5, update_freq='epoch'),
        LRSchedulerLogger(log_dir2),
    ],
    verbose = 0
)
print(f"Cosine LR run written to: {log_dir2}")

## Comparing Two Runs in TensorBoard

When `--logdir logs/` is used, TensorBoard shows all child runs together. Compare Adam with constant LR vs Adam with cosine decay.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
epochs_range = range(1, 61)
ax.plot(epochs_range, history.history['val_loss'],  label='Adam constant LR',  lw=1.5)
ax.plot(epochs_range, hist2.history['val_loss'],    label='Adam cosine LR',    lw=1.5, linestyle='--')
ax.set_xlabel('Epoch'); ax.set_ylabel('Validation MSE')
ax.set_title('Run comparison — same chart shown in TensorBoard Scalars')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print("In TensorBoard: open Scalars tab, select both runs — curves overlay automatically.")

## Projector: Log Intermediate Embeddings

Extract the hidden_3 layer's 16-dimensional activations for 500 test samples. Write them as TSV files for the TensorBoard Projector tab.

In [ ]:
# ── Extract layer 3 activations ───────────────────────────────────────────────
embedding_model = keras.Model(
    inputs  = model.input,
    outputs = model.get_layer('hidden_3').output
)
n_emb      = 500
embeddings = embedding_model.predict(X_te[:n_emb], verbose=0)   # (500, 16)
print(f"Embedding tensor shape: {embeddings.shape}")

# ── Write TSV files ───────────────────────────────────────────────────────────
proj_dir = os.path.join(log_dir, 'projector')
os.makedirs(proj_dir, exist_ok=True)

np.savetxt(os.path.join(proj_dir, 'feature_vecs.tsv'),
           embeddings, delimiter='\t', fmt='%.6f')

# Metadata: house value quantile (low / mid / high) for colouring
y_te_flat = y_te[:n_emb, 0]
q33, q67  = np.percentile(y_te_flat, [33, 67])
with open(os.path.join(proj_dir, 'metadata.tsv'), 'w') as f:
    f.write('MedHouseVal\tValueBand\n')
    for v in y_te_flat:
        band = 'Low' if v < q33 else ('High' if v > q67 else 'Mid')
        f.write(f'{v:.3f}\t{band}\n')

# Write projector_config.pbtxt
config_txt = f"""embeddings {{
  tensor_path: "feature_vecs.tsv"
  metadata_path: "metadata.tsv"
}}"""
with open(os.path.join(proj_dir, 'projector_config.pbtxt'), 'w') as f:
    f.write(config_txt)

print(f"Projector files written to: {proj_dir}")
print("In TensorBoard: open Projector tab → load config from this directory.")

## In-Notebook: 2D PCA of the Embedding (Projector Preview)

Reproduce what TensorBoard's Projector tab shows with PCA, directly in the notebook.

In [ ]:
from sklearn.decomposition import PCA

pca2    = PCA(n_components=2, random_state=42)
emb_2d  = pca2.fit_transform(embeddings)
y_colour = y_te_flat

plt.figure(figsize=(8, 6))
sc = plt.scatter(emb_2d[:, 0], emb_2d[:, 1], c=y_colour, cmap='viridis', s=10, alpha=0.7)
plt.colorbar(sc, label='MedHouseVal ($100k)')
plt.xlabel(f'PC1 ({pca2.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'PC2 ({pca2.explained_variance_ratio_[1]*100:.1f}%)')
plt.title('PCA of hidden_3 activations — colour=MedHouseVal\n(matches TensorBoard Projector tab)')
plt.tight_layout(); plt.show()

print("If house values are smoothly ordered in this 2D space,")
print("the network has learned a useful internal representation.")

## Diagnosing Vanishing Gradients via Weight Changes

Simulate what TensorBoard's Histograms panel reveals: compare the weight mean/std at epoch 1 vs epoch 60 for each layer.

In [ ]:
# ── Record weights at start and end of training ───────────────────────────────
model_diag = build_model()

# Weights before any training
weights_before = {layer.name: layer.get_weights()[0].copy()
                  for layer in model_diag.layers if layer.get_weights()}

model_diag.fit(X_tr, y_tr, epochs=60, batch_size=256, verbose=0)

# Weights after training
weights_after = {layer.name: layer.get_weights()[0]
                 for layer in model_diag.layers if layer.get_weights()}

print(f"{'Layer':<12}  {'Before std':>12}  {'After std':>12}  {'Δ std':>12}")
for name in weights_before:
    s_b = weights_before[name].std()
    s_a = weights_after[name].std()
    print(f"{name:<12}  {s_b:>12.6f}  {s_a:>12.6f}  {s_a - s_b:>+12.6f}")

print("\nA layer where Δ std ≈ 0 across many epochs → likely vanishing gradients")

## What Can Go Wrong: Logging Every Batch

Compare log directory sizes and event counts when using `update_freq='epoch'` vs `update_freq='batch'`.

In [ ]:
import shutil

for freq, label in [('epoch', 'epoch_log'), ('batch', 'batch_log')]:
    log_tmp  = os.path.join('logs', f'ch16_freq_test_{label}')
    m_tmp    = build_model()
    m_tmp.fit(
        X_tr, y_tr,
        epochs          = 5,
        batch_size      = 256,
        callbacks       = [keras.callbacks.TensorBoard(
            log_dir=log_tmp, update_freq=freq, histogram_freq=0)],
        verbose = 0
    )
    size_bytes = sum(
        os.path.getsize(os.path.join(dp, f))
        for dp, _, files in os.walk(log_tmp) for f in files
    )
    n_events = 5 if freq == 'epoch' else (5 * (len(X_tr) // 256))
    print(f"update_freq='{freq:>5}': {size_bytes/1024:.1f} KB   ~{n_events} scalar entries per metric")

print("\n→ 'batch' creates ~60× more events — unusable for long runs")

## Exercises

1. **Dead neuron detection.** Modify `build_model()` to use **sigmoid** activations instead of ReLU. Train for 60 epochs and compare the hidden_1 weight std before and after training to the ReLU version. Does sigmoid show more evidence of frozen early-layer weights?

2. **Gradient norm logging.** Write a custom Keras callback that computes the L2 norm of gradients for each layer at the end of each epoch (hint: use `tf.GradientTape`) and logs them as TensorBoard scalars. Plot the gradient norms across epochs for each layer. Do the norms decrease in early layers relative to the output layer?

3. **Profiler tab.** Re-run training with `profile_batch=2` and open the TensorBoard Profile tab. Which operations dominate the per-step time? Is the bottleneck in the forward pass, backward pass, or data loading?

In [ ]:
# Exercise 1 — Dead neuron detection with sigmoid
# TODO: your solution here

In [ ]:
# Exercise 2 — Gradient norm logger
# TODO: your solution here

In [ ]:
# Exercise 3 — TensorBoard Profiler
# TODO: your solution here